# Sprint 5 Runner (Colab)

<!-- Cell 0: Judul notebook - intro Sprint 5 Plan v3 -->

Notebook khusus untuk menjalankan `Sprint 5 Plan v3` end-to-end dengan output log tampil penuh di setiap sel.

## 0) Settings

<!-- Cell 1: Header section Settings -->

Isi `REPO_URL` dengan repository kamu. Kalau repo private, pastikan token/Git auth sudah siap di Colab.

In [1]:
# Cell 2: Core settings - REPO_URL, branch, path, stage toggles
REPO_URL = 'https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git'  # contoh: https://github.com/<user>/<repo>.git
REPO_BRANCH = 'feat/sprint5-cse-fpr-reduction'  # branch yang mau dipakai di Colab
PROJECT_NAME = 'nids-cnn-lstm-autoencoder'
DRIVE_ROOT = '/content/drive/MyDrive/nids-cnn-lstm-autoencoder'

# Raw dataset source (default mengikuti Sprint 3)
# Jika folder ini tidak ada, notebook fallback ke {DRIVE_ROOT}/data/raw
RAW_DRIVE_SOURCE = '/content/drive/MyDrive/nids-data/raw'

FORCE_RECLONE = False

# Stage toggles (set True/False sesuai kebutuhan)
RUN_STAGE0 = True
RUN_STAGE1 = True
RUN_STAGE2 = False
RUN_STAGE3_4 = True
RUN_SUMMARIZE = True


In [2]:
# Cell 3: Colab bootstrap + mount Google Drive
import os
import sys
import json
import time
import shlex
import shutil
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


IN_COLAB = True
Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/nids-cnn-lstm-autoencoder


In [3]:
# Cell 4: Clone atau update repo di /content + checkout branch
PROJECT_ROOT = Path('/content') / PROJECT_NAME

if FORCE_RECLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    if '<REPO_URL_HERE>' in REPO_URL:
        raise ValueError('Set REPO_URL dulu di cell Settings')
    cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

# Always sync and checkout selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--prune'], check=False)

# Try checkout branch; if branch only exists on origin, create tracking local branch.
ret = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_BRANCH], check=False)
if ret.returncode != 0:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '-b', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)

# Pull latest on selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', REPO_BRANCH], check=False)

active_branch = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], text=True).strip()
print('[GIT] active branch =', active_branch)


[CMD] git clone https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git /content/nids-cnn-lstm-autoencoder
PROJECT_ROOT = /content/nids-cnn-lstm-autoencoder
cwd = /content/nids-cnn-lstm-autoencoder
[GIT] active branch = feat/sprint5-cse-fpr-reduction


In [4]:
# Cell 5: Install dependencies (Colab-safe, skip reinstall numpy/tensorflow)
from pathlib import Path
import importlib.util

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')


def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if IN_COLAB:
    # IMPORTANT:
    # Colab sudah punya numpy/pandas/scikit/tensorflow yang saling kompatibel.
    # Reinstall paket-paket ini sering memicu ABI mismatch (numpy.dtype size changed).
    print('[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).')

    # Install only lightweight utilities if missing.
    lightweight = [
        'pyyaml',
        'joblib',
        'seaborn',
    ]

    for pkg in lightweight:
        mod = 'yaml' if pkg == 'pyyaml' else pkg
        if importlib.util.find_spec(mod) is None:
            _run([sys.executable, '-m', 'pip', 'install', pkg])
        else:
            print(f'[INFO] {pkg} already available')

    print('\n[OK] Dependency step finished (Colab-safe mode).')
    print('[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.')
else:
    # Local/non-Colab: follow project requirements as usual.
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req)])


[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).
[INFO] pyyaml already available
[INFO] joblib already available
[INFO] seaborn already available

[OK] Dependency step finished (Colab-safe mode).
[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.


In [5]:
# Cell 6: Link data/model/results sprint5 ke Drive (persist setelah disconnect)
os.chdir(PROJECT_ROOT)

raw_source = RAW_DRIVE_SOURCE if Path(RAW_DRIVE_SOURCE).exists() else f'{DRIVE_ROOT}/data/raw'
print(f'[RAW] using source: {raw_source}')

paths = [
    ('data/raw', raw_source),
    ('data/sprint5', f'{DRIVE_ROOT}/data/sprint5'),
    ('models/sprint5', f'{DRIVE_ROOT}/models/sprint5'),
    ('results/sprint5', f'{DRIVE_ROOT}/results/sprint5'),
]

for _, dst in paths:
    Path(dst).mkdir(parents=True, exist_ok=True)

for src, dst in paths:
    src_path = Path(src)
    if src_path.is_symlink() or src_path.exists():
        if src_path.is_symlink() or src_path.is_file():
            src_path.unlink()
        else:
            shutil.rmtree(src_path)
    src_path.parent.mkdir(parents=True, exist_ok=True)
    src_path.symlink_to(Path(dst), target_is_directory=True)
    print(f'[LINK] {src} -> {dst}')

print('[OK] Symlink setup complete')

# Fast preflight check
required_dirs = [
    Path('data/raw/CIC-IDS2017'),
    Path('data/raw/CSE-CIC-IDS2018'),
]
for d in required_dirs:
    print(f'[CHECK] {d}:', 'OK' if d.exists() else 'MISSING')


[RAW] using source: /content/drive/MyDrive/nids-data/raw
[LINK] data/raw -> /content/drive/MyDrive/nids-data/raw
[LINK] data/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/data/sprint5
[LINK] models/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/models/sprint5
[LINK] results/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/results/sprint5
[OK] Symlink setup complete
[CHECK] data/raw/CIC-IDS2017: OK
[CHECK] data/raw/CSE-CIC-IDS2018: OK


In [6]:
# Cell 7: GPU check + run_cmd_stream helper (streaming output ke cell)
try:
    import tensorflow as tf
except Exception as e:
    print('[ERROR] TensorFlow import gagal:', repr(e))
    print('Kemungkinan besar environment ABI belum sinkron setelah pip install.')
    print('Solusi: Runtime > Restart runtime, lalu jalankan lagi dari cell GPU check ini.')
    raise

print('Python executable :', sys.executable)
print('TensorFlow        :', tf.__version__)
print('Built with CUDA   :', tf.test.is_built_with_cuda())
print('Built with GPU sup:', tf.test.is_built_with_gpu_support())
print('Physical GPU list :', tf.config.list_physical_devices('GPU'))
print('Logical GPU list  :', tf.config.list_logical_devices('GPU'))

# Maksimalkan penggunaan GPU: memory growth agar TF pakai memori dinamis
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
if gpus:
    print('[GPU] Memory growth enabled -> training/eval akan memaksimalkan GPU.')


def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'


def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts


def run_cmd_stream(title: str, cmd: str, log_to_file: bool = True):
    """Jalankan command dengan streaming output ke cell. Log lengkap disimpan ke file."""
    print(f'[RUN] {title}', flush=True)
    args = _normalize_cmd(cmd)
    print('[CMD]', ' '.join(args), flush=True)
    t0 = time.time()

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        args,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    log_path = None
    log_file = None
    if log_to_file:
        log_dir = PROJECT_ROOT / 'results' / 'sprint5' / 'runtime'
        log_dir.mkdir(parents=True, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
        log_path = log_dir / f'log_{title.replace(" ", "_")}_{ts}.txt'
        log_file = open(log_path, 'w', encoding='utf-8')

    last_line = ''
    try:
        for line in proc.stdout or []:
            print(line, end='', flush=True)
            if log_file:
                log_file.write(line)
                log_file.flush()
            last_line = line.strip()
    finally:
        if log_file:
            log_file.close()
            print(f'\n[LOG] Full log saved to: {log_path}', flush=True)

    ret = proc.wait()
    print(f'\n[EXIT CODE] {ret}', flush=True)
    if ret != 0:
        raise RuntimeError(f"{title} failed (exit={ret}). Last line: {last_line}")

    print(f'[DONE] {title} in {_fmt_duration(time.time() - t0)}', flush=True)


def run_stage(stage_name: str):
    run_cmd_stream(
        title=f'Sprint4 {stage_name}',
        cmd=f'python scripts/sprint5/research_runner.py --stage-names {stage_name} --skip-existing',
    )


Python executable : /usr/bin/python3
TensorFlow        : 2.19.0
Built with CUDA   : True
Built with GPU sup: True
Physical GPU list : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPU list  : [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
[GPU] Memory growth enabled -> training/eval akan memaksimalkan GPU.


## 1) Optional Dry-Run

<!-- Cell 8: Header dry-run - validasi tanpa eksekusi real -->

Validasi command tanpa eksekusi real training/eval.

In [7]:
# Cell 9: Execute dry-run stage0 (validasi command tanpa training/eval real)
run_cmd_stream(
    title='Sprint4 Dry-Run Stage0',
    cmd='python scripts/sprint5/research_runner.py --dry-run --stage-names stage0 --no-summarize',
)


[RUN] Sprint4 Dry-Run Stage0
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --dry-run --stage-names stage0 --no-summarize
[RUN] s5_00_lock_baseline | stage=stage0 | stages=['preprocess', 'train', 'eval'] | tag=s5_00_lock_baseline
[CMD] /usr/bin/python3 scripts/sprint5/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml
[CMD] /usr/bin/python3 scripts/sprint5/train.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml --variant hybrid
[CMD] /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml --model models/sprint5/s5_00_lock_baseline/cnn_lstm_ae/best_model.keras --tag s5_00_lock_baseline
[RUN] s5_m07_rerun | stage=stage0 | stages=['train', 'eval'] | tag=s5_m07_rerun
[CMD] /usr/bin/python3 scripts/sprint5/train.py --config /content/nids-cnn-lstm-autoencoder/research/spr

## 2) Execute Sprint 5 Stages

<!-- Cell 10: Header execute stages -->

In [8]:
# Cell 11: Execute stage0 (repro + strict contract runs)
if RUN_STAGE0:
    run_stage('stage0')
else:
    print('[SKIP] stage0')


[RUN] Sprint4 stage0
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage0 --skip-existing
[SKIP] s5_00_lock_baseline metrics already exist for tag=s5_00_lock_baseline
[SKIP] s5_m07_rerun metrics already exist for tag=s5_m07_rerun
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/log_Sprint4_stage0_20260301_072224.txt

[EXIT CODE] 0
[DONE] Sprint4 stage0 in 1s


In [9]:
# Cell 12: Execute stage1 (7 target threshold variant runs)
if RUN_STAGE1:
    run_stage('stage1')
else:
    print('[SKIP] stage1')


[RUN] Sprint4 stage1
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage1 --skip-existing
[RUN] s5_t01_target_p95 | stage=stage1 | stages=['eval'] | tag=s5_t01_target_p95
[CMD] /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_t01_target_p95.yaml --model models/sprint5/s5_00_lock_baseline/cnn_lstm_ae/best_model.keras --tag s5_t01_target_p95
2026-03-01 07:22:28.518662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772349748.541223    4119 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772349748.549312    4119 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

In [10]:
# Cell 13: Execute stage2 (Sprint 5: no stage2, skip)
if RUN_STAGE2:
    run_stage('stage2')
else:
    print('[SKIP] stage2')


[SKIP] stage2


In [11]:
# Cell 14: Execute stage3 + stage4 (guardrail relax + seed candidates)
if RUN_STAGE3_4:
    run_cmd_stream(
        title='Sprint4 stage3+stage4',
        cmd='python scripts/sprint5/research_runner.py --stage-names stage3,stage4 --skip-existing',
    )
else:
    print('[SKIP] stage3,stage4')


[RUN] Sprint4 stage3+stage4
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage3,stage4 --skip-existing
[SKIP] s5_h01_guardrail_fpr012 | Run s5_h01_guardrail_fpr012 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s5_h02_guardrail_fpr015 | Run s5_h02_guardrail_fpr015 requires template_from_best_stage=stage1, but source stage is inconclusive.
[SKIP] s5_c01_seed42 gate_pass condition not met
[SKIP] s5_c02_seed1234 gate_pass condition not met
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/log_Sprint4_stage3+stage4_20260301_072329.txt

[EXIT CODE] 0
[DONE] Sprint4 stage3+stage4 in 0s


In [12]:
# Cell 15: Execute summarize-only (aggregate hasil, gate decision, report)
if RUN_SUMMARIZE:
    run_cmd_stream(
        title='Sprint4 summarize-only',
        cmd='python scripts/sprint5/research_runner.py --summarize-only',
    )
else:
    print('[SKIP] summarize-only')


[RUN] Sprint4 summarize-only
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --summarize-only
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/log_Sprint4_summarize-only_20260301_072330.txt

[EXIT CODE] 0
[DONE] Sprint4 summarize-only in 0s


## 3) Show Outputs (Summary, Gate, Report)

<!-- Cell 16: Header show outputs -->

In [13]:
# Cell 17: Tampilkan summary.csv, gate_decision.json, report markdown
import pandas as pd
from IPython.display import display, Markdown

summary_path = PROJECT_ROOT / 'results/sprint5/summary.csv'
gate_path = PROJECT_ROOT / 'results/sprint5/gate_decision.json'
report_path = PROJECT_ROOT / 'docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT4.md'
status_path = PROJECT_ROOT / 'results/sprint5/runtime/run_status.json'

print('summary_path =', summary_path)
print('gate_path    =', gate_path)
print('report_path  =', report_path)
print('status_path  =', status_path)

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print('\n[SUMMARY HEAD]')
    display(df.head(30))
    print('\n[STATUS COUNTS]')
    if 'terminal_status' in df.columns:
        display(df['terminal_status'].value_counts(dropna=False))
else:
    print('[WARN] summary.csv not found')

if gate_path.exists():
    gate = json.loads(gate_path.read_text(encoding='utf-8'))
    print('\n[GATE DECISION]')
    print(json.dumps(gate, indent=2))
else:
    print('[WARN] gate_decision.json not found')

if report_path.exists():
    print('\n[REPORT PREVIEW]')
    txt = report_path.read_text(encoding='utf-8')
    display(Markdown(txt[:8000]))
else:
    print('[WARN] report markdown not found')

if status_path.exists():
    print('\n[RUNTIME STATUS JSON]')
    print(status_path.read_text(encoding='utf-8')[:8000])


summary_path = /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
gate_path    = /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json
report_path  = /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT4.md
status_path  = /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/run_status.json

[SUMMARY HEAD]


,run_id,stage_name,profile,model_variant,active,condition,stages,tag,terminal_status,retry_count,...,cse_prec,cse_rec,cse_f1,cse_fpr,cse_auc,f1_gap,auc_gap,accuracy_gap,wall_time_sec,last_error
0,s5_00_lock_baseline,stage0,s5_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s5_00_lock_baseline,success,0,...,0.874147,0.625923,0.729498,0.223683,NaN,0.102882,0.0,0.098824,2030.676788,NaN
1,s5_m07_rerun,stage0,s5_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s5_m07_rerun,success,0,...,0.885812,0.698368,0.781001,0.223459,NaN,0.046242,0.0,0.040696,1895.925668,NaN
2,s5_t01_target_p95,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t01_target_p95,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 13.30s: /usr/bin/pyt...
3,s5_t02_target_p97,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t02_target_p97,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.30s: /usr/bin/pyth...
4,s5_t03_target_p98,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t03_target_p98,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.40s: /usr/bin/pyth...
5,s5_t04_target_p99,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t04_target_p99,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.34s: /usr/bin/pyth...
6,s5_t05_target_p97_sub5pct,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t05_target_p97_sub5pct,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.34s: /usr/bin/pyth...
7,s5_t06_target_p97_sub10pct,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t06_target_p97_sub10pct,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.36s: /usr/bin/pyth...
8,s5_t07_source_calib_guardrail,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t07_source_calib_guardrail,failed_experiment,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Command failed rc=1 after 8.36s: /usr/bin/pyth...
9,s5_h01_guardrail_fpr012,stage3,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_h01_guardrail_fpr012,skipped_inconclusive,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,Run s5_h01_guardrail_fpr012 requires template_...



[STATUS COUNTS]


,count
terminal_status,
failed_experiment,7
success,2
skipped_inconclusive,2
skipped_gate,2



[GATE DECISION]
{
  "generated_at": "2026-03-01T07:23:30.787001",
  "gate_pass": false,
  "reason": "stage3_no_candidate",
  "adaptive_recall_target": 0.75,
  "adaptive_target_reasoning": "fixed_floor_only",
  "best_stage3_run_id": null,
  "best_stage3_metrics": null,
  "pivot_recommendation": "USAD",
  "stage_validity": {
    "stage1": {
      "required": 7,
      "valid_count": 0,
      "passed": false,
      "status": "inconclusive"
    },
    "stage2": {
      "required": 0,
      "valid_count": 0,
      "passed": true,
      "status": "ok"
    },
    "stage3": {
      "required": 2,
      "valid_count": 0,
      "passed": false,
      "status": "inconclusive"
    },
    "stage4": {
      "required": 2,
      "valid_count": 0,
      "passed": true,
      "status": "skipped_by_design"
    }
  },
  "sprint_history_summary": {
    "recall_p50": 0.6621453946312135,
    "recall_p90": 0.691123311245271,
    "best_recall_under_guardrail": null,
    "best_recall_path": "",
    "count_all"

## 4) Resume Guide

<!-- Cell 18: Panduan resume setelah runtime putus -->

Kalau runtime Colab putus:
1. Run lagi dari cell mount + clone/pull + symlink.
2. Jalankan stage berikutnya atau stage yang sama.
3. `--skip-existing` akan melanjutkan dari artifact yang sudah ada.